# 06 — Prompts, fake client, matrix, experiment log

The last offline slice: local JSON-strict prompt variants, the
deterministic fake OpenAI client used by `--mock` pipeline runs, a
provider×model×prompt matrix **dry-run**, and the sandbox-local
experiment log (not a mirror of llm-entity-extraction).

Canonical code:

- [`src/mailroom_sandbox/prompts.py`](../src/mailroom_sandbox/prompts.py)
- [`config/prompts/`](../config/prompts)
- [`src/mailroom_sandbox/mock_llm.py`](../src/mailroom_sandbox/mock_llm.py)
- [`src/mailroom_sandbox/eval/matrix.py`](../src/mailroom_sandbox/eval/matrix.py)
- [`src/mailroom_sandbox/eval/experiment_log.py`](../src/mailroom_sandbox/eval/experiment_log.py)


In [ ]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    """Walk up from cwd (hostile kernels start in notebooks/)."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "reports").is_dir():
            return cand
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

from _lib import bootstrap, isolate_outputs

ROOT = bootstrap(ROOT)
OUT = isolate_outputs(ROOT)
print("repo root :", ROOT)
print("sys.path[0]:", sys.path[0])
print("notebook outputs ->", OUT)


## Local prompt variants (7B/8B JSON-strict)


In [ ]:
from mailroom_sandbox.prompts import list_variants, load_variant, variant_path

print("variants:", list_variants())
text = load_variant("sorter_local_v0")
print("path:", variant_path("sorter_local_v0"))
print("--- sorter_local_v0 (head) ---")
print(text[:500])
assert "json" in text.lower()
print()
print("CLI: sandbox eval sorter --mock --prompt sorter_local_v0")


## Fake client: classify vs extract vs ambiguous confidence


In [ ]:
from mailroom_sandbox.mock_llm import fake_structured_payload

classify = fake_structured_payload(
    "Classify this legal document into one of the mailroom classes.",
    {"doc_type": "contract", "id": "contract_msa"},
)
amb = fake_structured_payload(
    "Classify this legal document into one of the mailroom classes.",
    {"doc_type": "correspondence", "id": "ambiguous_01"},
)
extract = fake_structured_payload(
    "Extract the parties and dates from this MSA.",
    {"doc_type": "contract", "expected_fields": {"parties": ["Acme"], "effective_date": "2024-01-01"}},
)
print("classify :", classify)
print("ambiguous confidence (routing REVIEW case):", amb["confidence"])
print("extract  :", extract)
assert amb["confidence"] == 0.40
assert classify["confidence"] == 0.97


## Matrix dry-run (no LLM, no writes except this notebook's log later)


In [ ]:
from mailroom_sandbox.eval.matrix import plan_matrix, run_matrix

plan = run_matrix(
    task="sorter",
    providers=["ollama"],
    models=["qwen3:8b", "llama3.2:3b"],
    prompts=["mailroom-default", "sorter_local_v0"],
    sample=2,
    mock=True,
    dry_run=True,
)
print("n cells:", plan["n"])
for cell in plan["cells"]:
    print(" ", cell["experiment_name"], "model=", cell["model"], "prompt=", cell["prompt"])
assert plan["n"] == 4
print()
print("CLI: sandbox matrix --providers ollama --models qwen3:8b --prompts sorter_local_v0 --mock --dry-run")


## Experiment log (sandbox-local JSONL)


In [ ]:
from mailroom_sandbox.eval.runners import run_isolated_eval
from mailroom_sandbox.eval import experiment_log

run_isolated_eval("sorter", mock=True, sample=2, experiment_name="nb06_log_probe")
records = experiment_log.load()
mine = [r for r in records if str(r.get("experiment_name", "")).startswith("nb06_")]
print("notebook log path:", experiment_log.jsonl_path())
print("nb06 records:", len(mine))
if mine:
    rec = mine[-1]
    print("keys:", sorted(rec.keys()))
    print("task/profile/mock:", rec.get("task"), rec.get("profile"), rec.get("mock"))
    print("scores:", rec.get("scores"))
    print("sandbox flag:", rec.get("sandbox"), "(this is NOT llm-entity-extraction's log)")
print()
print("Markdown sibling is regenerated on every append:", experiment_log.md_path())
